# Using GraphX in Apache

        This is a place holder[NC 2024-11-10]
This notebook uses pyspark and graphframes. REMEMBER that any serious work (in terms of scale) should be done in the native language -- Scala.
        

In [1]:
# code from https://www.edureka.co/blog/spark-graphx/ translated to python

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Create a SparkSession
# Note: how do you know which package to download?
#  Try without --> get error --> search for it --> find the package name 
#  goto "maven repository" --> search for the package name without postfix ('graphframes')
#  Select the version you want. By default, pick the latest.
#  Choose the
spark = SparkSession.builder.appName('basicGraph')\
    .config("spark.kryoserializer.buffer.max", "512m")\
    .config("spark.jars.packages", 'graphframes:graphframes:0.8.1-spark3.0-s_2.12')\
    .getOrCreate()

In [2]:
# Create a DataFrame from vertex and edge data
vertex_df = spark.createDataFrame([
    (1, "Alice", 30),
    (2, "Bob", 25),
    (3, "Charlie", 35),
    (4, "David", 42),
    (5, "Ed", 55)
], ["id", "name", "age"])

edge_df = spark.createDataFrame([
    (1, 2),
    (1, 3),
    (2, 3)
], ["src", "dst"])

# Create a GraphFrame
from graphframes import GraphFrame
g = GraphFrame(vertex_df, edge_df)

# Filter vertices based on age
filtered_vertices = g.vertices.filter(col("age") > 30)

# Print the filtered vertices
filtered_vertices.show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  3|Charlie| 35|
|  4|  David| 42|
|  5|     Ed| 55|
+---+-------+---+



# Count in-degree (e.g. number of followers in LinkedIn)

In [8]:
from pyspark.sql.functions import coalesce, lit

followers = spark.createDataFrame([
    ("1", "2", "follow"),
    ("2", "3", "follow"),
    ("3", "5", "follow"),
    ("4", "5", "follow"),
    ("5", "1", "follow"),
    ("1", "3", "follow")
], ["src", "dst", "relationship"])


initial_user_graph = GraphFrame(vertex_df, followers)

# Calculate in-degrees and out-degrees
in_degrees = initial_user_graph.inDegrees.withColumnRenamed("inDegree", "inDeg")
out_degrees = initial_user_graph.outDegrees.withColumnRenamed("outDegree", "outDeg")

# Join degree information with the user vertices
user_graph = initial_user_graph.vertices \
    .join(in_degrees, on="id", how="left") \
    .join(out_degrees, on="id", how="left") \
    .withColumn("inDeg", coalesce("inDeg", lit(0))) \
    .withColumn("outDeg", coalesce("outDeg", lit(0)))

# Show the final graph properties
user_graph.show()

# Print user information
for row in user_graph.collect():
    print(f"User {row['id']} is called {row['name']} and is liked by {row['inDeg']} people.")

+---+-------+---+-----+------+
| id|   name|age|inDeg|outDeg|
+---+-------+---+-----+------+
|  1|  Alice| 30|    1|     2|
|  2|    Bob| 25|    1|     1|
|  3|Charlie| 35|    2|     1|
|  4|  David| 42|    0|     1|
|  5|     Ed| 55|    2|     1|
+---+-------+---+-----+------+

User 1 is called Alice and is liked by 1 people.
User 2 is called Bob and is liked by 1 people.
User 3 is called Charlie and is liked by 2 people.
User 4 is called David and is liked by 0 people.
User 5 is called Ed and is liked by 2 people.


In [23]:
vertex_df.show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  2|    Bob| 25|
|  3|Charlie| 35|
|  4|  David| 42|
|  5|     Ed| 55|
+---+-------+---+



In [22]:
# Find the oldest follower for each user
oldest_follower = vertex_df.groupBy("id")
# Show results
oldest_follower.agg({"age": "min"}).show()

+---+--------+
| id|min(age)|
+---+--------+
|  1|      30|
|  2|      25|
|  3|      35|
|  4|      42|
|  5|      55|
+---+--------+



https://spark.apache.org/docs/latest/graphx-programming-guide.html#structural-operators

In [16]:
# Find the oldest follower for each user
oldest_follower = vertex_df.groupBy("id") \
    .agg({"age": "max"}) \
    .join(vertex_df, ["id", "max(age)"]) \
    .select("id", "name", "age")

# Show results
oldest_follower.show()

AnalysisException: USING column `max(age)` cannot be resolved on the right side of the join. The right-side columns: [id, name, age]

In [ ]:
load("../data/graphs/sibling.parqeut")